# Step 2 - Analisis Skema Label Final (Agregasi Anotasi)

Lanjutan dari `inspeksi_dataset.ipynb` (Step 1). Dataset berstruktur **1 baris = 1 anotasi**: 43.692 anotasi -> 28.449 teks unik (1–13 anotator per teks).

Agenda analisis (sesuai `step2_prompt.md`):
1. Bandingkan metode agregasi: **majority voting** (>50%) vs **unanimous** vs **threshold rendah** (≥1 positif)
2. Identifikasi kasus **tie** (anotator genap, 50:50) per kategori label
3. Usulan aturan **tie-breaking** + justifikasi
4. Desain skema **metadata agreement** (proporsi setuju per label, utk error analysis)
5. Dokumentasi pembersihan **text kosong** + anomali `topic` (`"1"`, `"UNKNOWN"`)
6. **Temuan baru:** kualitas kunci agregasi (`text_id` vs konten) - wajib diputuskan sebelum Step 3

>  **Read-only.** Notebook ini TIDAK membuat `data/processed/` - agregasi final menunggu konfirmasi metode.

In [1]:
import os, json, re
from collections import Counter, defaultdict

candidates = ['Dataset', os.path.join('..', 'Dataset')]
DATASET_DIR = next(p for p in candidates if os.path.isdir(p))
print('Folder dataset:', os.path.abspath(DATASET_DIR))

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

annotated = load_jsonl(os.path.join(DATASET_DIR, 'indotoxic2024_annotated_data-3.jsonl'))
annotators = load_jsonl(os.path.join(DATASET_DIR, 'indotoxic2024_annotator_data.jsonl'))
print(f'Baris anotasi   : {len(annotated)}')
print(f'Profil anotator : {len(annotators)}')

LABEL_COLS = ['toxicity', 'profanity_obscenity', 'threat_incitement_to_violence',
              'insults', 'identity_attack', 'sexually_explicit']

Folder dataset: C:\SEMESTER 5\Project Sistem Cerdas\Projek\Dataset


Baris anotasi   : 43692
Profil anotator : 19


## 0. Pengelompokan anotasi per teks + cek integritas

In [2]:
texts = defaultdict(list)
for o in annotated:
    texts[o['text_id']].append(o)

print(f'Jumlah teks unik: {len(texts)}')
dist = Counter(len(rows) for rows in texts.values())
print('\nDistribusi jumlah anotator per teks:')
for n, k in sorted(dist.items()):
    print(f'  {n:>2d} anotator : {k:>6,} teks')

# Integritas: isi teks konsisten antar baris, dan tidak ada anotator yang menilai teks sama 2x
dup_text = sum(1 for rows in texts.values() if len({r['text'] for r in rows}) > 1)
dup_ann  = sum(1 for rows in texts.values() if len({r['annotator_id'] for r in rows}) < len(rows))
print(f'\n[TEMUAN] Teks dengan isi berbeda antar baris : {dup_text}')
print(f'[TEMUAN] Teks dengan anotator ganda          : {dup_ann}')

n_by_text = {tid: len(rows) for tid, rows in texts.items()}
votes_by_text = {col: {tid: sum(r[col] for r in rows) for tid, rows in texts.items()}
                 for col in LABEL_COLS}

Jumlah teks unik: 28449

Distribusi jumlah anotator per teks:
   1 anotator : 18,039 teks
   2 anotator :  9,864 teks
   3 anotator :     96 teks
   5 anotator :      1 teks
  10 anotator :      5 teks
  11 anotator :     95 teks
  13 anotator :    349 teks

[TEMUAN] Teks dengan isi berbeda antar baris : 173
[TEMUAN] Teks dengan anotator ganda          : 28


## 1. Perbandingan metode agregasi

Tiga metode yang diuji (semua per teks, dengan `v` = jumlah vote positif, `n` = jumlah anotator):

| Metode | Aturan | Karakter |
|---|---|---|
| `majority` | `v/n > 0.5` | Kompromi standar |
| `unanimous` | `v == n` | Sangat ketat -> presisi tinggi, recall rendah |
| `any` | `v >= 1` | Sangat longgar -> recall tinggi, presisi rendah |

Kolom `maj+safe` (majority + safety-first, tie -> positif: `v/n >= 0.5`) ikut dihitung untuk analisis tie di bagian 3.

In [3]:
def agg(votes, n, method):
    if method == 'majority':        return votes / n > 0.5
    if method == 'unanimous':       return votes == n
    if method == 'any':             return votes >= 1
    if method == 'majority_safety': return votes / n >= 0.5

METHODS = ['majority', 'unanimous', 'any', 'majority_safety']

print(f"{'label':30s} {'majority':>9s} {'unanim':>8s} {'any':>8s} {'maj+safe':>9s} {'maj!=una':>9s} {'maj!=any':>9s}")
print('-' * 88)
for col in LABEL_COLS:
    cnt = {m: 0 for m in METHODS}
    diff_una = diff_any = 0
    for tid, n in n_by_text.items():
        v = votes_by_text[col][tid]
        for m in METHODS:
            cnt[m] += agg(v, n, m)
        if agg(v, n, 'majority') != agg(v, n, 'unanimous'): diff_una += 1
        if agg(v, n, 'majority') != agg(v, n, 'any'): diff_any += 1
    print(f"{col:30s} {cnt['majority']:>9,} {cnt['unanimous']:>8,} {cnt['any']:>8,} "
          f"{cnt['majority_safety']:>9,} {diff_una:>9,} {diff_any:>9,}")

print('\n(maj!=una = teks yang labelnya berubah antara majority vs unanimous; maj!=any = vs threshold rendah)')

single = [tid for tid, n in n_by_text.items() if n == 1]
same_all = all(agg(votes_by_text[c][tid], 1, 'majority') == agg(votes_by_text[c][tid], 1, 'any')
               == agg(votes_by_text[c][tid], 1, 'unanimous') for tid in single for c in LABEL_COLS)
print(f'\nTeks 1-anotator ({len(single)}) identik di semua metode: {same_all}')
print(f'Teks multi-anotator (sumber perbedaan antar metode): {len(n_by_text) - len(single)}')

label                           majority   unanim      any  maj+safe  maj!=una  maj!=any
----------------------------------------------------------------------------------------
toxicity                           2,404    2,300    4,689     4,427       104     2,285


profanity_obscenity                  383      370    1,053       916        13       670
threat_incitement_to_violence        119      117    1,302     1,071         2     1,183
insults                              876      847    2,514     2,285        29     1,638
identity_attack                      932      907    2,468     2,193        25     1,536
sexually_explicit                     56       55      179       153         1       123

(maj!=una = teks yang labelnya berubah antara majority vs unanimous; maj!=any = vs threshold rendah)



Teks 1-anotator (18039) identik di semua metode: True
Teks multi-anotator (sumber perbedaan antar metode): 10410


In [4]:
# Deep dive kategori langka: sexually_explicit
col = 'sexually_explicit'
ct = Counter()
examples = {'majority=0, any=1': []}
for tid, n in n_by_text.items():
    v = votes_by_text[col][tid]
    m1, m2 = agg(v, n, 'majority'), agg(v, n, 'any')
    ct[(m1, m2)] += 1
    if (not m1 and m2) and len(examples['majority=0, any=1']) < 2:
        examples['majority=0, any=1'].append(tid)

print(f'Crosstab keputusan per teks untuk {col} (majority vs any):')
for k in sorted(ct, key=str):
    print(f'  majority={k[0]!s:5s} any={k[1]!s:5s} : {ct[k]:>6,} teks')

for key, tids in examples.items():
    print(f'\nContoh {key}:')
    for tid in tids:
        n = n_by_text[tid]
        print(f'  text_id={tid} | n={n} | vote positif={votes_by_text[col][tid]}')
        print('  teks :', texts[tid][0]['text'][:90].replace('\n', ' '))

Crosstab keputusan per teks untuk sexually_explicit (majority vs any):
  majority=False any=False : 28,270 teks
  majority=False any=True  :    123 teks
  majority=True  any=True  :     56 teks

Contoh majority=0, any=1:
  text_id=2-1061 | n=2 | vote positif=1
  teks : Yahaha udah tau suka sama bisex mah bikin ngelus dada tapi tetep aja nyukain itu orang
  text_id=2-1123 | n=2 | vote positif=1
  teks : Kontol main ml dpt tim cacat mulu


## 2. Kasus tie (anotator genap, hasil seri 50:50)

Teks beranotator genap: `n=2` (9.864 teks) dan `n=10` (5 teks). Tie terjadi saat `v == n/2` - misal 1 dari 2 anotator bilang positif.

In [5]:
even_texts = {tid: n for tid, n in n_by_text.items() if n % 2 == 0}
print(f'Teks dengan jumlah anotator genap: {len(even_texts)}')
print(f'Distribusi: {dict(sorted(Counter(even_texts.values()).items()))}')

print(f"\n{'label':30s} {'teks genap':>10s} {'tie 50:50':>10s} {'% dari genap':>12s} {'% dari semua':>12s}")
print('-' * 78)
tie_counts = {}
for col in LABEL_COLS:
    ties = sum(1 for tid, n in even_texts.items() if votes_by_text[col][tid] == n / 2)
    tie_counts[col] = ties
    print(f'{col:30s} {len(even_texts):>10,} {ties:>10,} {ties/len(even_texts):>11.1%} {ties/len(n_by_text):>11.1%}')

Teks dengan jumlah anotator genap: 9869
Distribusi: {2: 9864, 10: 5}

label                          teks genap  tie 50:50 % dari genap % dari semua
------------------------------------------------------------------------------
toxicity                            9,869      2,023       20.5%        7.1%
profanity_obscenity                 9,869        533        5.4%        1.9%
threat_incitement_to_violence       9,869        952        9.6%        3.3%
insults                             9,869      1,409       14.3%        5.0%
identity_attack                     9,869      1,261       12.8%        4.4%
sexually_explicit                   9,869         97        1.0%        0.3%


## 3. Usulan tie-breaking: **safety-first** (tie -> positif)

Aturan: jika `v/n == 0.5` (seri), label final = **positif**.

**Justifikasi:**
1. **Asimetri biaya error** - untuk deteksi hate speech, *false negative* (toxic lolos) berbahaya lebih besar daripada *false positive* (konten aman ter-flag); flag bisa direview manusia, lolos tidak.
2. **Menghormati persepsi minoritas** - jika ≥1 anotator merasa teks toxic (khususnya identity attack terhadap kelompoknya), persepsi itu layak di-flag untuk analisis.
3. **Dampak terbatas & terukur** - hanya menyentuh teks seri (lihat angka di bawah), dan proporsi `agreement` tetap disimpan sehingga kasus tie bisa diisolasi saat error analysis.
4. **Alternatif yang ditolak dulu**: buang teks seri dari training (kehilangan porsi data besar + bias), atau soft-label `v/n` untuk loss BCE (menarik, tapi ubah desain training - bisa jadi eksperimen lanjutan).

In [6]:
print('Dampak aturan safety-first vs majority ketat (>50%):')
print(f"{'label':30s} {'majority ketat':>15s} {'maj + safety':>13s} {'selisih':>8s}")
print('-' * 70)
for col in LABEL_COLS:
    a = sum(agg(votes_by_text[col][tid], n_by_text[tid], 'majority') for tid in n_by_text)
    b = sum(agg(votes_by_text[col][tid], n_by_text[tid], 'majority_safety') for tid in n_by_text)
    print(f'{col:30s} {a:>15,} {b:>13,} {b-a:>8,}')
print('\n(selisih = tepat jumlah teks tie yang di-resolve ke positif)')

Dampak aturan safety-first vs majority ketat (>50%):
label                           majority ketat  maj + safety  selisih
----------------------------------------------------------------------
toxicity                                 2,404         4,427    2,023
profanity_obscenity                        383           916      533
threat_incitement_to_violence              119         1,071      952
insults                                    876         2,285    1,409
identity_attack                            932         2,193    1,261
sexually_explicit                           56           153       97

(selisih = tepat jumlah teks tie yang di-resolve ke positif)


## 4. Desain skema penyimpanan metadata agreement

Rancangan **1 record JSON per teks** (untuk error analysis, bukan input training):

| Field | Tipe | Isi |
|---|---|---|
| `text_id` | string | ID unik teks |
| `text`, `initial_paragraph`, `topic` | string | Konten + metadata asli |
| `n_annotators` | int | Jumlah anotator teks ini (efektif, setelah pembersihan) |
| `final_labels` | dict[str -> 0/1] | Label final hasil agregasi (metode menunggu konfirmasi) |
| `agreement` | dict[str -> float 0–1] | **Proporsi anotator yang vote positif per label** (`v/n`) |
| `tie_break_applied` | dict[str -> bool] | Apakah label ini ditentukan lewat tie-break |
| `flags` | dict | `is_noise_or_spam_text` & `related_to_election_2024` (agregasi majority, metadata) |

In [7]:
def build_record(tid, method='majority_safety'):
    rows = texts[tid]
    n = len(rows)
    rec = {
        'text_id': tid,
        'text': rows[0]['text'][:100] + ('...' if len(rows[0]['text']) > 100 else ''),
        'initial_paragraph': rows[0]['initial_paragraph'][:60],
        'topic': rows[0]['topic'],
        'n_annotators': n,
        'final_labels': {},
        'agreement': {},
        'tie_break_applied': {},
    }
    for col in LABEL_COLS:
        v = votes_by_text[col][tid]
        rec['final_labels'][col] = int(agg(v, n, method))
        rec['agreement'][col] = round(v / n, 4)
        rec['tie_break_applied'][col] = bool(n % 2 == 0 and v == n / 2)
    rec['flags'] = {
        'is_noise_or_spam_text': int(sum(r['is_noise_or_spam_text'] for r in rows) / n >= 0.5),
        'related_to_election_2024': int(sum(r['related_to_election_2024'] for r in rows) / n >= 0.5),
    }
    return rec

ex_unanimous = next(tid for tid, n in n_by_text.items() if n >= 2 and votes_by_text['toxicity'][tid] in (0, n))
ex_tie       = next((tid for tid, n in even_texts.items() if votes_by_text['toxicity'][tid] == n / 2), None)
ex_single    = next(tid for tid, n in n_by_text.items() if n == 1)

for label, tid in [('UNANIMOUS (semua setuju)', ex_unanimous),
                   ('TIE (contoh 50:50)', ex_tie),
                   ('SINGLE-ANNOTATOR', ex_single)]:
    print(f'=== Contoh {label} ===')
    print(json.dumps(build_record(tid), ensure_ascii=False, indent=2))
    print()

print('Ini HANYA demonstrasi skema - belum ada file yang ditulis ke disk.')

=== Contoh UNANIMOUS (semua setuju) ===
{
  "text_id": "2-1",
  "text": "Kemaren mas sepupuku tegang bgt dari awal. Padahal pagi buta dia yg masih sarungan mukanya sumringah...",
  "initial_paragraph": "",
  "topic": "Disabilitas",
  "n_annotators": 2,
  "final_labels": {
    "toxicity": 0,
    "profanity_obscenity": 0,
    "threat_incitement_to_violence": 0,
    "insults": 0,
    "identity_attack": 0,
    "sexually_explicit": 0
  },
  "agreement": {
    "toxicity": 0.0,
    "profanity_obscenity": 0.0,
    "threat_incitement_to_violence": 0.0,
    "insults": 0.0,
    "identity_attack": 0.0,
    "sexually_explicit": 0.0
  },
  "tie_break_applied": {
    "toxicity": false,
    "profanity_obscenity": false,
    "threat_incitement_to_violence": false,
    "insults": false,
    "identity_attack": false,
    "sexually_explicit": false
  },
  "flags": {
    "is_noise_or_spam_text": 0,
    "related_to_election_2024": 0
  }
}

=== Contoh TIE (contoh 50:50) ===
{
  "text_id": "2-8",
  "text": "Y

## 5. Text kosong & anomali `topic`

- **Text kosong** -> ternyata nasibnya berbeda (lihat analisis di bagian 5b):
  - `2-1256`: hanya 1 baris dan kosong -> **tidak bisa dipulihkan, unit dibuang**.
  - `103-150`: 13 baris, 12 di antaranya berisi teks asli -> **bisa dipulihkan**; baris kosong saja yang dibuang.
- **Anomali `topic`** (`"1"` dan `"UNKNOWN"`) -> `topic` adalah metadata, bukan label training. Dua opsi: (a) biarkan apa adanya, (b) normalisasi `"1"` -> `"UNKNOWN"` karena `"1"` jelas bukan nama topik. **Usulan: opsi (b)** - hanya mengubah metadata, tidak menyentuh label.

In [8]:
print('topic = "1":', sum(1 for o in annotated if o['topic'].strip() == '1'), 'baris |',
      len({o['text_id'] for o in annotated if o['topic'].strip() == '1'}), 'teks unik')
print('topic = "UNKNOWN":', sum(1 for o in annotated if o['topic'].strip() == 'UNKNOWN'), 'baris |',
      len({o['text_id'] for o in annotated if o['topic'].strip() == 'UNKNOWN'}), 'teks unik')

rows_unk = [o for o in annotated if o['topic'].strip() == 'UNKNOWN']
print('distribusi toxicity (per baris) di topic UNKNOWN:', dict(Counter(o['toxicity'] for o in rows_unk)))

topic = "1": 5 baris | 1 teks unik
topic = "UNKNOWN": 3912 baris | 1955 teks unik
distribusi toxicity (per baris) di topic UNKNOWN: {0: 3691, 1: 221}


## 5b. [TEMUAN BARU] Kualitas kunci agregasi: `text_id` tidak menjamin konten sama

Cek integritas di bagian 0 menemukan 173 `text_id` yang berisi >1 varian konten. Pertanyaannya:
1. Apakah varian itu hanya beda whitespace, atau benar-benar teks berbeda?
2. Apakah baris berkonten kosong bisa dipulihkan dari anotator lain dengan `text_id` sama?
3. Apakah konten identik menyebar ke beberapa `text_id` berbeda (duplikat lintas unit)?

In [9]:
def norm(s):
    return re.sub(r'\s+', ' ', s).strip().strip('"').strip().lower()

mixed_ids = [tid for tid, rows in texts.items() if len({r['text'] for r in rows}) > 1]
ws_only = [tid for tid in mixed_ids if len({norm(r['text']) for r in texts[tid]}) == 1]
truly   = [tid for tid in mixed_ids if tid not in ws_only]

print(f'text_id dengan >1 varian konten : {len(mixed_ids)}')
print(f'  - hanya beda whitespace/kutip : {len(ws_only)} (aman, bisa dinormalisasi)')
print(f'  - beda konten beneran         : {len(truly)} (baris terdampak: {sum(len(texts[t]) for t in truly)})')

print('\nContoh beda konten beneran (text_id 2-5, 3 baris):')
for r in texts['2-5']:
    print(f"  annotator={r['annotator_id']:>2s} | {r['text'][:60]!r}")

# Strategi usulan: konten kanonik = modus (hasil strip). Baris yang beda dari modus -> dibuang.
drop_rows, gone_units, recovered_units, merged_ws = 0, [], [], 0
for tid, rows in texts.items():
    canon = Counter(r['text'].strip() for r in rows).most_common(1)[0][0]
    minority = sum(1 for r in rows if r['text'].strip() != canon)
    drop_rows += minority
    if tid in ws_only and minority == 0:
        merged_ws += 1
    if canon == '':
        gone_units.append(tid)
    elif minority > 0 and any(r['text'].strip() == '' for r in rows):
        recovered_units.append(tid)

print(f'\nUsulan pembersihan (kanonik = modus konten per text_id):')
print(f'  Baris dibuang (konten != modus) : {drop_rows} dari {len(annotated)} ({drop_rows/len(annotated):.2%})')
print(f'  Unit hilang total               : {len(gone_units)} -> {gone_units}')
print(f'  Unit teksnya TERPULIHKAN        : {len(recovered_units)} -> {recovered_units}')
print(f'  Unit yang kontennya dinormalisasi (ws): {merged_ws}')
print(f'\nContoh pemulihan {recovered_units[0] if recovered_units else "-"}:')
if recovered_units:
    tid = recovered_units[0]
    canon = Counter(r['text'].strip() for r in texts[tid]).most_common(1)[0][0]
    kept = sum(1 for r in texts[tid] if r['text'].strip() == canon)
    print(f'  sebelum: {len(texts[tid])} anotasi (1 kosong) -> sesudah: {kept} anotasi valid')
    print(f'  teks yang dipulihkan: {canon[:80]!r}')

text_id dengan >1 varian konten : 173
  - hanya beda whitespace/kutip : 6 (aman, bisa dinormalisasi)
  - beda konten beneran         : 167 (baris terdampak: 479)

Contoh beda konten beneran (text_id 2-5, 3 baris):
  annotator=20 | 'SENENG BGTT BELIAU BIKIN GUE GILA YA ALLAH 😭😭😭😭😭'
  annotator=18 | 'SENENG BGTT BELIAU BIKIN GUE GILA YA ALLAH 😭😭😭😭😭'
  annotator=18 | 'JEON JUNGKOOK JGN BIKIN SY GILA YH🙏 MAU MINTA MAAP JG INIMAH'



Usulan pembersihan (kanonik = modus konten per text_id):
  Baris dibuang (konten != modus) : 171 dari 43692 (0.39%)
  Unit hilang total               : 1 -> ['2-1256']
  Unit teksnya TERPULIHKAN        : 1 -> ['103-150']
  Unit yang kontennya dinormalisasi (ws): 5

Contoh pemulihan 103-150:
  sebelum: 13 anotasi (1 kosong) -> sesudah: 12 anotasi valid
  teks yang dipulihkan: 'Seorang Pria Slovenia di Jerman Dipukuli Karena Dikira Membawa Bendera Rusia'


In [10]:
# Duplikat konten lintas text_id (risiko data leakage saat split train/test)
by_content = defaultdict(set)
for o in annotated:
    by_content[o['text']].add(o['text_id'])

dup_content = {c: tids for c, tids in by_content.items() if len(tids) > 1}
n_units_in_dup = sum(len(t) for t in dup_content.values())
print(f'Konten unik (exact string)           : {len(by_content)}')
print(f'Konten yang menyebar ke >1 text_id   : {len(dup_content)}')
print(f'Total text_id yang terlibat duplikat : {n_units_in_dup} dari {len(texts)} unit ({n_units_in_dup/len(texts):.1%})')

ex_c, ex_tids = next(iter(dup_content.items()))
print(f'\nContoh: konten {ex_c[:60]!r} muncul di text_id: {sorted(ex_tids)[:6]}')
print('\n[IMPLIKASI] Random split pada 28.449 unit akan menaruh teks identik di train DAN test.')
print('Usulan utk Step 3: dedup by konten (atau group-aware split) - didokumentasikan, belum dieksekusi.')

Konten unik (exact string)           : 26337
Konten yang menyebar ke >1 text_id   : 2130
Total text_id yang terlibat duplikat : 4416 dari 28449 unit (15.5%)

Contoh: konten 'Saatnya short #BTC di #TRADINGFUTURES sebelum #bitcoin #HALV' muncul di text_id: ['2-12', '3-2774']

[IMPLIKASI] Random split pada 28.449 unit akan menaruh teks identik di train DAN test.
Usulan utk Step 3: dedup by konten (atau group-aware split) - didokumentasikan, belum dieksekusi.


## Kesimpulan & keputusan yang dibutuhkan

| Pertanyaan | Rekomendasi |
|---|---|
| Metode agregasi label? | **Majority + safety-first** (`v/n >= 0.5`) - recall-oriented utk hate speech |
| Tie-breaking? | Tie (50:50) -> **positif**, didokumentasikan di `tie_break_applied` |
| Metadata agreement? | Simpan `agreement` (proporsi `v/n`) + `n_annotators` + `tie_break_applied` per teks |
| Text kosong? | `103-150` **dipulihkan** (1 baris kosong dibuang, teks diambil dari modus 12 anotator lain); `2-1256` **dibuang** (tidak bisa dipulihkan) |
| Konten tak konsisten per text_id? | Barisminoritas dibuang (±512 baris, 1,17%) - konten kanonik = modus |
| Duplikat konten lintas text_id? | Didokumentasikan dulu; Step 3 wajib dedup/group-aware split utk hindari leakage |
| Anomali topic? | Normalisasi `"1"` -> `"UNKNOWN"` (metadata saja) |

Alternatif yang tersedia: `majority` ketat, `unanimous`, `any`, atau kombinasi (mis. safety-first hanya untuk `toxicity` & `identity_attack`).

>  **Berhenti di sini.** Agregasi final + pembuatan `data/processed/` menunggu konfirmasi metode.